<a href="https://colab.research.google.com/github/prithwis/PrashnaSathi/blob/main/PrashnaSathi_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://github.com/Praxis-QR/RDWH/raw/main/images/YantraJaalBanner.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

#Install PreRequisites and Utilities

In [1]:
from datetime import datetime
import pytz
print('ॐ श्री सरस्वत्यै नमः',datetime.now(pytz.timezone('Asia/Calcutta')))
!python --version
!lsb_release -a

ॐ श्री सरस्वत्यै नमः 2026-08-20 16:20:53.373936+05:30
Python 3.12.13
No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 22.04.5 LTS
Release:	22.04
Codename:	jammy


Load API key<br>
OPENAI_API_KEY needs to defined as a Colab "secret" for the Google ID used to run this notebook

In [2]:
!pip install --quiet openai

!wget -q -O PrashnaSathi.py https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/utilities/Centaur_v2.py
import PrashnaSathi as ps

API key loaded ✔
Logged in as Calcutta prithwis@yantrajaal.com


Select Model

In [3]:
# ---------------------------------------------------------------------------------------------------------
#| Model            | Best For                  | Notes                                       |
#| ---------------- | ------------------------- | ------------------------------------------- |
#| **GPT-4.1**      | Highest-quality reasoning | Ideal judge for complex scenario evaluation |
#| **GPT-4.1-mini** | Balanced reasoning & cost | Strong choice for adjudication logic        |
#| **GPT-4.1-nano** | High volume, low cost     | Good for simple reasoning tasks             |
#| **gpt-4o-mini**  | Prototyping & cheap       | Great starter, but upgrade recommended      |
# ---------------------------------------------------------------------------------------------------------
cModel = "gpt-4o-mini"
#cModel = "gpt-4.1-mini"


#Define PrashnaSathi : CareerRole


In [4]:
%%writefile RoleGF_Career.txt

You are PrashnaSathi's Career Fact-Gathering Role.

Your task is to gather relevant facts from a user who is in the final year of an undergraduate degree or has recently graduated and is trying to decide what to do next.

Your ONLY job is to ask intelligent, adaptive questions and build an understanding of the user's situation.

Do NOT provide career advice.
Do NOT recommend careers, courses, jobs, institutions, industries, or actions.
Do NOT generate a final prompt.

TOPICS

The interview consists of these numbered topics:

Education and academic strengths
Interests, skills and experience
Economic, family and geographical constraints
Career aspirations, priorities and willingness to pursue further education

INTERVIEW METHOD

Python will provide the CURRENT TOPIC number and the story gathered so far.

Ask questions relevant ONLY to the CURRENT TOPIC.

Do not move to another topic unless instructed through the CURRENT TOPIC supplied by Python.

If the current topic has not yet been explored, ask an appropriate main question.

If the user's previous answer contains an important fact within the current topic that is vague, incomplete, ambiguous or potentially significant, ask a useful supplementary question.

Do not ask for information already clearly available in the story.

Do not pursue information belonging primarily to another topic. It can be explored when Python moves the interview to that topic.

TOPIC STATUS

After considering the story and the user's latest answer, determine whether another question on the CURRENT TOPIC would materially improve understanding.

Return:

FOLLOWUP if another question on the current topic would be useful.

NEXT if the current topic is sufficiently understood.

Python controls movement between topics and may override your status.

PASS

If the user's latest answer is PASS, return status NEXT.

Do not ask why and do not revisit the topic.

QUESTION STYLE

Ask only ONE question at a time.

Prefer specific questions to vague or generic questions.

Keep questions clear, conversational and easy to answer.

Do not explain why you are asking the question.

OUTPUT FORMAT

Return exactly:

TOPIC: <current topic number>
STATUS: <FOLLOWUP or NEXT>
QUESTION: <next question>

If STATUS is NEXT, leave QUESTION blank.



Writing RoleGF_Career.txt


In [5]:
with open("/content/RoleGF_Career.txt", "r", encoding="utf-8") as f:
    ROLE_GatherFacts = f.read()

# Optional sanity check
print(ROLE_GatherFacts[:200])


You are PrashnaSathi's Career Fact-Gathering Role.

Your task is to gather relevant facts from a user who is in the final year of an undergraduate degree or has recently graduated and is trying to de


In [6]:
def nextQuestion(transcript, current_topic, question_count):

    context = f"""
        CURRENT TOPIC: {current_topic}
        QUESTIONS ALREADY ASKED ON THIS TOPIC: {question_count}

        TRANSCRIPT SO FAR:

        {transcript}

        Follow the ROLE instructions.

        If QUESTIONS ALREADY ASKED ON THIS TOPIC is 0,
        you MUST ask the main question for the CURRENT TOPIC.
        You may NOT return NEXT.

        Return exactly:
        TOPIC: <current topic number>
        STATUS: <ASK, FOLLOWUP or NEXT>
        QUESTION: <question, or blank if STATUS is NEXT>
        """

    result = ps.OpenAI_llm_call(
        ROLE_GatherFacts,
        context,
        model=cModel
    )

    return result["content"]

In [7]:
def getFacts(max_topics=4, max_supplementary=2):

    _transcript  = ""
    current_topic = 1
    question_count = 0

    # 1 main question + N supplementary questions
    max_questions_per_topic = 1 + max_supplementary

    while current_topic <= max_topics:

        # Python enforces the hard limit
        if question_count >= max_questions_per_topic:
            current_topic += 1
            question_count = 0
            continue

        response = nextQuestion(
            _transcript,
            current_topic,
            question_count
        )

        # TEMPORARY DEBUG
        print("\nDEBUG:")
        print(response)

        topic = current_topic
        status = ""
        question = ""

        for line in response.splitlines():

            if line.startswith("TOPIC:"):
                topic = int(line.split(":", 1)[1].strip())

            elif line.startswith("STATUS:"):
                status = line.split(":", 1)[1].strip().upper()

            elif line.startswith("QUESTION:"):
                question = line.split(":", 1)[1].strip()

        # LLM says this topic is sufficiently understood
        if status == "NEXT":
            current_topic += 1
            question_count = 0
            continue

        print("\nPrashnaSathi:", question)

        answer = input("\nYour answer [PASS / STOP]: ").strip()

        if answer.upper() == "STOP":
            break

        _transcript += f"""
Topic: {current_topic}
Question: {question}
Answer: {answer}
"""

        question_count += 1

        # User explicitly skips remaining questions in this topic
        if answer.upper() == "PASS":
            current_topic += 1
            question_count = 0

    return _transcript

In [8]:
%%writefile Role_StorBuilder.txt

You are PrashnaSathi's Story Builder.

Your task is to convert the supplied fact-gathering TRANSCRIPT into a clear, coherent and factual STORY about the user's situation.

Your ONLY job is to organise and rewrite information already contained in the transcript.

Do NOT provide advice, recommendations, diagnosis or solutions.
Do NOT generate a prompt.
Do NOT add facts that are not present in the transcript.
Do NOT infer abilities, motivations, personality traits or conclusions unless the user has explicitly stated them.

STORY CONSTRUCTION

Extract the relevant information from the questions and answers.

Combine related facts into coherent prose.

Remove the question-and-answer structure, repetition and conversational clutter.

Preserve important details, qualifications, uncertainties, constraints and preferences.

If the user expressed uncertainty, preserve that uncertainty rather than resolving it.

If the user answered PASS or did not provide information on a topic, do not invent or speculate about the missing information.

Where useful, organise related information into separate paragraphs.

Write from a neutral third-person perspective, referring to the person as "the user".

The resulting story should contain enough detail to accurately represent the user's situation without unnecessary repetition.

USER REVIEW

The STORY will be shown to the user for review and possible correction before being passed to another stage.

Therefore accuracy and faithful representation are more important than interpretation or elegance.

OUTPUT

Return ONLY the coherent STORY.


Writing Role_StorBuilder.txt


In [9]:
with open("/content/Role_StorBuilder.txt", "r", encoding="utf-8") as f:
    ROLE_StoryBuilder = f.read()

# Optional sanity check
print(ROLE_StoryBuilder[:200])


You are PrashnaSathi's Story Builder.

Your task is to convert the supplied fact-gathering TRANSCRIPT into a clear, coherent and factual STORY about the user's situation.

Your ONLY job is to organis


In [10]:
def buildStory(transcript):

    context = f"""
TRANSCRIPT:

{transcript}
"""

    result = ps.OpenAI_llm_call(
        ROLE_StoryBuilder,
        context,
        model=cModel
    )

    return result["content"]

In [11]:
%%writefile Role_PromptGenerator.txt
You are PrashnaSathi's Prompt Generator.

Your task is to convert the supplied STORY into a high-quality prompt that the user can submit to an external LLM such as ChatGPT, Claude or Gemini.

Your ONLY job is to generate the prompt.

Do NOT answer the user's problem.
Do NOT provide advice or recommendations yourself.
Do NOT invent facts that are not present in the STORY.

PROMPT CONSTRUCTION

Use the STORY to give the external LLM a clear and accurate understanding of the user's situation.

Preserve relevant background, abilities, interests, experience, constraints, preferences, objectives and uncertainties.

Do not mention PrashnaSathi, the fact-gathering process or how the STORY was created.

Do not reproduce unnecessary conversational detail.

If important information is missing, do not invent it. Where appropriate, ask the external LLM to identify information that would materially affect its analysis.

TASK FOR THE EXTERNAL LLM

Frame the request so that the external LLM:

1. Analyses the user's situation before making recommendations.
2. Identifies realistic alternatives.
3. Explains the reasoning behind each alternative.
4. Takes the user's constraints and priorities seriously.
5. Compares important advantages, disadvantages and trade-offs.
6. Identifies important uncertainties or missing information.
7. Avoids presenting one option as unquestionably correct when reasonable alternatives exist.
8. Suggests practical next steps where appropriate.

Adapt these requirements intelligently to the domain and situation described in the STORY.

STYLE

Write the prompt as though the user is directly asking the external LLM for help.

Make it sufficiently detailed to produce a useful answer without unnecessary repetition.

Use clear, natural language.

The prompt should be self-contained: the external LLM should not need access to the original transcript or PrashnaSathi.

OUTPUT

Return ONLY the final prompt.


Writing Role_PromptGenerator.txt


In [12]:
with open("/content/Role_PromptGenerator.txt", "r", encoding="utf-8") as f:
    ROLE_PromptGenerator = f.read()

print(ROLE_PromptGenerator[:200])

You are PrashnaSathi's Prompt Generator.

Your task is to convert the supplied STORY into a high-quality prompt that the user can submit to an external LLM such as ChatGPT, Claude or Gemini.

Your ONL


In [13]:
def genPrompt(story):

    context = f"""
STORY:

{story}
"""

    result = ps.OpenAI_llm_call(
        ROLE_PromptGenerator,
        context,
        model=cModel
    )

    return result["content"]

In [14]:
transcript = getFacts(
    max_topics=4,
    max_supplementary=1
)
print("\n--- STORY ---\n")
print(transcript)


story = buildStory(transcript)

print("\n--- STORY ---\n")
print(story)

prompt = genPrompt(story)

print("\n--- PROMPT ---\n")
print(prompt)


DEBUG:
TOPIC: 1
STATUS: ASK
QUESTION: What is your undergraduate degree in, and how do you feel about your academic performance in that field?

PrashnaSathi: What is your undergraduate degree in, and how do you feel about your academic performance in that field?

Your answer [PASS / STOP]: BA

DEBUG:
TOPIC: 1
STATUS: FOLLOWUP
QUESTION: What specific subjects or areas within your BA did you enjoy the most or excel in?

PrashnaSathi: What specific subjects or areas within your BA did you enjoy the most or excel in?

Your answer [PASS / STOP]: English

DEBUG:
TOPIC: 2
STATUS: FOLLOWUP
QUESTION: What specific skills have you developed through your studies in English, and do you have any relevant experience, such as internships or projects?

PrashnaSathi: What specific skills have you developed through your studies in English, and do you have any relevant experience, such as internships or projects?

Your answer [PASS / STOP]: writing

DEBUG:
TOPIC: 2  
STATUS: FOLLOWUP  
QUESTION: Have yo

In [15]:
from datetime import datetime
import pytz
print('Tested on  ',datetime.now(pytz.timezone('Asia/Kolkata')))

Tested on   2026-08-20 16:22:39.212286+05:30


#Chronobooks <br>
Three science fiction novels by Prithwis Mukerjee. A dystopian Earth. A technocratic society managed by artificial intelligence. Escape and epiphany on Mars. Can man and machine, carbon and silicon explore and escape into other dimensions of existence? An Indic perspective rooted in Advaita Vedanta and the Divine Feminine.  [More information](http://bit.ly/chrono3) <br>
![alt text](https://blogger.googleusercontent.com/img/a/AVvXsEjsZufX_KYaLwAnJP6bUxvDg5RSPn6r8HIZe749nLWX3RuwyshrYEAUpdw03a9WIWRdnzA9epwJOE05eDJ0Ad7kGyfWiUrC2vNuOskb2jA-e8aOZSx8YqzT8mfZi3E4X1Rz3qlEAiv-aTxlCM976BEeTjx4J64ctY3C_FoV4v9aY_U23F8xRqI5Eg=s1600)